# SQL Agent

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 실습 환경 준비

In [3]:
import pathlib
import requests

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} 파일이 이미 존재합니다. 다운로드를 건너뜁니다.")
else:
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"{local_path} 파일 다운로드 완료")
    else:
        print(f"다운로드 실패. status code: {response.status_code}")

Chinook.db 파일이 이미 존재합니다. 다운로드를 건너뜁니다.


In [4]:
from langchain_community.utilities import SQLDatabase

# sqlite:/// 는 로컬 SQLite 파일에 연결할 때 사용하는 URI 형식이다.
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"SQL dialect: {db.dialect}")
print(f"사용 가능한 테이블 목록: {db.get_usable_table_names()}")

SQL dialect: sqlite
사용 가능한 테이블 목록: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


## SQL Agent가 사용할 도구 만들기

In [6]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.chat_models import init_chat_model 

model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()

for tool in tools:
    print(f'[{tool.name}]')
    print(tool.description)
    print()

[sql_db_query]
Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

[sql_db_schema]
Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

[sql_db_list_tables]
Input is an empty string, output is a comma-separated list of tables in the database.

[sql_db_query_checker]
Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [7]:
# 이후 노드 구성에서 특정 도구만 꺼내쓰기 위해 선언해 둔다.
list_tables_tool = next(tool for tool in tools if tool.name == 'sql_db_list_tables')
get_schema_tool = next(tool for tool in tools if tool.name == 'sql_db_schema')
run_query_tool = next(tool for tool in tools if tool.name == 'sql_db_query')
query_checker_tool = next(tool for tool in tools if tool.name == 'sql_db_query_checker')

## LangGraph로 작업 단계 나누기
1. 테이블 목록을 조회한다.
2. 질문 해결에 필요한 테이블의 스키마를 조회한다.
3. 사용자 질문에 맞는 SQL을 생성한다.
4. 생성 된 SQL을 실행 전에 다시 검토한다.
5. 검토 된 SQL을 실행한다.
6. 실행 결과를 바탕으로 최종 답변을 생성한다.

In [9]:
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

### 1. 테이블 목록 조회 노드
첫 번째 노드는 항상 테이블 목록을 조회한다.

In [10]:
def list_tables(state: MessagesState):
    """데이터베이스의 테이블 목록을 조회하는 노드이다."""

    tool_call = {
        "name" : "sql_db_list_tables",
        "args" : {},
        "id" : "list_tables_call",
        "type" : "tool_call"
    }

    # AI가 이 도구를 호출하려고 했다는 기록을 남기기 위해서
    tool_call_messages = AIMessage(content='', tool_calls=[tool_call])

    # 실제 테이블 목록 조회 도구 실행
    tool_message = list_tables_tool.invoke(tool_call)

    # 이후 LLM이 읽기 쉽도록 테이블 목록을 일반 AIMessage로도 추가
    response = AIMessage(content=f'사용 가능한 테이블 목록 : {tool_message.content}')

    return {"messages" : [tool_call_messages, tool_message, response]}

In [11]:
# 노드 단위 테스트

state = {'messages' : [HumanMessage(content='평균 재생 시간이 가장 긴 장르는 무엇인가?')]}
response = list_tables(state)

for message in response['messages']:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (list_tables_call)
 Call ID: list_tables_call
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================

사용 가능한 테이블 목록 : Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


### 2. 스키마 조회 노드
질문 해결에 필요한 테이블의 스키마를 조회한다.

In [12]:
def call_get_schema(state: MessagesState):
    """사용자 질문과 테이블 목록을 보고 필요한 테이블 스키마 조회 도구를 호출한다."""

    # tool_choice='any'의 의미는 제공 된 tool 중 하나를 호출하는 것을 강제하는 의미
    llm_with_tools = model.bind_tools([get_schema_tool], tool_choice='any')
    response = llm_with_tools.invoke(state['messages'])

    return {'messages' : [response]}

In [13]:
# 노드 단위 테스트
# 위의 테스트에서 응답 받은 response에 담긴 messages 재사용

state_after_list_tables = {
    "messages" : [
        HumanMessage(content='평균 재생시간이 가장 긴 장르는 무엇인가?'),
        *response['messages']
    ]
}

schema_call_response = call_get_schema(state_after_list_tables)
schema_call_response['messages'][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_yQ4E2qNtoZwWuJMlJf76ATp2)
 Call ID: call_yQ4E2qNtoZwWuJMlJf76ATp2
  Args:
    table_names: Genre, Track


### 3. SQL 생성 노드

In [ ]:
generate_query_system_prompt = f"""
당신은 SQL 데이터베이스와 상호작용하는 Agent입니다.

사용자의 질문을 보고 실행할 수 있는 올바른 {db.dialect} SQL 쿼리를 작성하세요.
사용자가 특정 개수를 명시하지 않으면 결과는 최대 5개로 제한하세요.
가장 의미 있는 결과가 먼저 나오도록 관련 컬럼 기준으로 정렬하세요.

주의사항:
- 특정 테이블의 모든 컬럼을 조회하지 마세요.
- 질문 해결에 필요한 컬럼만 조회하세요.
- INSERT, UPDATE, DELETE, DROP, ALTER, CREATE 같은 데이터 변경 또는 구조 변경 문장은 절대 만들지 마세요.
- SQL 실행이 필요하면 반드시 sql_db_query 도구를 호출하세요.
- 도구 실행 결과가 주어졌다면 그 결과를 바탕으로 한국어로 답변하세요.
"""

def generate_query(state: MessagesState):
    """질문 해결에 필요한 SQL을 생성하거나, 실행 결과를 바탕으로 최종 답변을 생성한다."""

    system_message = {
        "role": "system",
        "content": generate_query_system_prompt,
    }

    # 도구 호출을 강제하지 않는다.
    # SQL 실행이 필요하면 run_query_tool을 호출한다.    
    # 이미 SQL 실행 결과가 있다면 모델이 최종 답변을 생성한다.
    llm_with_tools = model.bind_tools([run_query_tool])
    response = llm_with_tools.invoke([system_message] + state["messages"])

    return {"messages": [response]}

### 4. SQL 검토 노드

In [ ]:
check_query_system_prompt = f"""
당신은 SQL 검토 전문가입니다.

다음 {db.dialect} SQL 쿼리를 실행하기 전에 꼼꼼히 검토하세요.

검토 기준:
- NOT IN과 NULL 값 조합으로 인한 문제
- UNION과 UNION ALL 사용 오류
- BETWEEN의 경계 포함 여부
- 조건절의 데이터 타입 불일치
- 식별자 quoting 문제
- 함수 인자 개수 오류
- CAST가 필요한 부분
- JOIN에 사용한 컬럼이 올바른지 여부
- 존재하지 않는 테이블이나 컬럼 사용 여부
- 데이터 변경 또는 구조 변경 문장 포함 여부

문제가 있으면 SQL을 수정하세요.
문제가 없으면 원래 SQL을 그대로 사용하세요.

검토가 끝나면 반드시 sql_db_query 도구를 호출하세요.
"""

def check_query(state: MessagesState):
    """생성된 SQL을 실행하기 전에 검토하고, 실행할 SQL 도구 호출을 다시 만든다."""

    system_message = {
        "role": "system",
        "content": check_query_system_prompt,
    }

    tool_call = state["messages"][-1].tool_calls[0]
    query = tool_call["args"]["query"]

    user_message = {"role": "user", "content": query}

    llm_with_tools = model.bind_tools([run_query_tool], tool_choice="any")
    response = llm_with_tools.invoke([system_message, user_message])

    response.id = state["messages"][-1].id

    return {"messages": [response]}